In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from IPython.display import display
import re

In [3]:
url = "https://www.rightmove.co.uk/commercial-property-for-sale/find.html?propertyTypes=hospitality%2Cleisure-facility%2Cbar-nightclub%2Ccafe%2Cguest-house%2Chotel%2Cpub%2Crestaurant%2Ctakeaway&sortType=13&areaSizeUnit=sqft&channel=COMMERCIAL_BUY&index=0&locationIdentifier=STATION%5E551&transactionType=BUY&displayLocationIdentifier=Bank-Station.html&radius=15.0&maxPrice=500000"
headers = {
    "User-Agent": "Mozilla//5.0",
    "Accept-languageA": "en-GB,en;q=0.9"
}
response = requests.get(url, headers=headers)
print("Status code:", response.status_code)

Status code: 200


In [4]:
# get the HTML content of the page
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")

full_links_check = soup.find_all("a", href=True)
for a in full_links_check:
    print(a.get("href"))

 

/
#
/property-for-sale.html
/new-homes-for-sale.html
/agent-valuation
/agent-valuation
/guides/landlord/investor-newsletter/
/mortgages
#
/property-to-rent.html
/student-accommodation.html
#
/house-prices.html
/agent-valuation
/agent-valuation
/house-value.html
#
/estate-agents
#
/commercial-property
/commercial-property?buy=true
/c/commercial-advertising/
#
/moving-stories/
/news/
/guides/energy-efficiency/
/guides/
/news/house-price-index/
/mortgages/guides/
/overseas-magazine/
/overseas-property/country-guides.html
#
/overseas-property.html
/overseas-property/in-Spain.html
/overseas-property/in-France.html
/overseas-property/in-Portugal.html
/overseas-property/in-Italy.html
/overseas-property/in-Greece.html
/overseas-magazine/currency-zone
/overseas-property/advertise.html
/
/login.html
/myrightmove.html
/property-for-sale.html
/new-homes-for-sale.html
/agent-valuation
/guides/landlord/investor-newsletter/
/mortgages
/property-to-rent.html
/student-accommodation.html
/estate-agents


In [5]:
print(soup.title.get_text(strip=True))

Commercial properties for sale near Bank Station | Rightmove


In [6]:
print(response.status_code)
print(response.url)
print(soup.title.get_text(strip=True))

200
https://www.rightmove.co.uk/commercial-property-for-sale/find.html?propertyTypes=hospitality%2Cleisure-facility%2Cbar-nightclub%2Ccafe%2Cguest-house%2Chotel%2Cpub%2Crestaurant%2Ctakeaway&sortType=13&areaSizeUnit=sqft&channel=COMMERCIAL_BUY&index=0&locationIdentifier=STATION%5E551&transactionType=BUY&displayLocationIdentifier=Bank-Station.html&radius=15.0&maxPrice=500000
Commercial properties for sale near Bank Station | Rightmove


In [7]:
property_tags = soup.select('a[href*="/properties/"]')

for tag in property_tags:
    property_tags = soup.select('a[href*="/properties/"]')

    for tag in property_tags:
        print(tag.get("href"))

/properties/174176258#/?channel=COM_BUY
/properties/174176258#/?channel=COM_BUY
/properties/174176258#/?channel=COM_BUY
/properties/174176258#/?channel=COM_BUY
/properties/174176258#/?channel=COM_BUY
/properties/174176258#/?channel=COM_BUY
/properties/174176258#/?channel=COM_BUY
/properties/174258068#/?channel=COM_BUY
/properties/174258068#/?channel=COM_BUY
/properties/174258068#/?channel=COM_BUY
/properties/174258068#/?channel=COM_BUY
/properties/174258068#/?channel=COM_BUY
/properties/174258068#/?channel=COM_BUY
/properties/174258068#/?channel=COM_BUY
/properties/152578379#/?channel=COM_BUY
/properties/152578379#/?channel=COM_BUY
/properties/152578379#/?channel=COM_BUY
/properties/152578379#/?channel=COM_BUY
/properties/152578379#/?channel=COM_BUY
/properties/152578379#/?channel=COM_BUY
/properties/152578379#/?channel=COM_BUY
/properties/87990147#/?channel=COM_BUY
/properties/87990147#/?channel=COM_BUY
/properties/87990147#/?channel=COM_BUY
/properties/87990147#/?channel=COM_BUY
/pro

In [8]:
data = []

a_tags = soup.find_all("div", attrs={"data-testid": re.compile(r"propertyCard-\d+")})

for a_tag in a_tags:
    # location
    address_tag = a_tag.find("address")
    location = address_tag.get_text(" ", strip=True) if address_tag else "N/A"

    # link
    link = "N/A"
    link_tags = a_tag.find_all("a", href=True)
    for a in link_tags:
        href = a["href"]
        if "/properties/" in href:
            link = "https://www.rightmove.co.uk" + href
            link = link.split("?")[0]
            link = link.split("#")[0]
            break

    # price
    price_tag = a_tag.find("div", class_="PropertyPrice_price__VL65t")
    price = price_tag.get_text(" ", strip=True) if price_tag else "N/A"

    # description
    desc_tag = a_tag.find("p", attrs={"data-testid": "property-description"})
    description = desc_tag.get_text(" ", strip=True) if desc_tag else "N/A"

    # nearest station
    station_tag = a_tag.find(attrs={"data-testid": "property-station-distance"})
    
    if station_tag:
        station_text = station_tag.get_text(" ", strip=True)

        match = re.search(r"\d+(\.\d+)?", station_text)

        if match:
            nearest_station = match.group()
        else:
            nearest_station = "N/A"
    else:
        nearest_station = "N/A"

    #size 
    size_tag = a_tag.find_all("div", class_="PropertyPrice_priceQualifier__U1Qu7")
    if len(size_tag) >= 2:
        size = size_tag[1].get_text(" ", strip=True)
    elif len(size_tag) == 1:
        size = size_tag[0].get_text(" ", strip=True)
    else:
        size = "N/A"
    
    size_check = size.lower()

    if size_check == "guide price" or size_check.startswith("offers in"):
        size = "N/A"
    #phone number
    phone_tag = a_tag.find("a", href=re.compile(r"tel:"))
    phone_number = phone_tag["href"].replace("tel:", "") if phone_tag else "N/A"
  
    # sale contact infomation 
    contact_tag = a_tag.find(attrs={"data-testid": "marketed-by-text"})
    contact_info = contact_tag.get_text(" ", strip=True) if contact_tag else "N/A"

    data.append({
        "location": location,
        "price": price,
        "size": size,
        "gap_station (miles from bank station)": nearest_station,
        "phone_number": phone_number,
        "contact_info": contact_info,
        "description": description,
        "links": link
    })
    df = pd.DataFrame(data)


In [9]:
detail_data = []

for link in df["links"]:
    sector = "N/A"
    key_features = "N/A"
    full_description = "N/A"
    nearest_stations_all = "N/A"

    if link != "N/A":
    
        detail_response = requests.get(link, headers=headers)
        detail_soup = BeautifulSoup(detail_response.text, "html.parser")

        detail_text = detail_soup.get_text("\n", strip=True)
        detail_lines = [line.strip() for line in detail_text.split("\n") if line.strip()]

        # sector
        if "SECTOR" in detail_lines:
            i_sector = detail_lines.index("SECTOR")
            if i_sector + 1 < len(detail_lines):
                sector = detail_lines[i_sector + 1]

            
        # key features
        if "Key features" in detail_lines:
            i_key_features = detail_lines.index("Key features") + 1
            features = []

            for line in detail_lines[i_key_features:]:
                if line in ["Description", "Brochures", "NEAREST STATIONS", "MARKETED BY"]:
                    break
                features.append(line)

            if len(features) > 0:
                key_features = " | ".join(features)

        # full description
        if "Description" in detail_lines:
            i_description = detail_lines.index("Description") + 1
            desc_parts = []

            for line in detail_lines[i_description:]:
                if line in ["Read full description", "Brochures", "NEAREST STATIONS", "MARKETED BY"]:
                    break
                desc_parts.append(line)

            if len(desc_parts) > 0:
                full_description = " ".join(desc_parts)

        # all nearest stations
        if "NEAREST STATIONS" in detail_lines:
            i_nearest_stations = detail_lines.index("NEAREST STATIONS") + 1
            stations = []

            j = i_nearest_stations
            while j < len(detail_lines):
                line = detail_lines[j]

                if line == "MARKETED BY" or line.startswith("About "):
                    break

                if "Station" in line and "miles" in line:
                    stations.append(line)

                elif "Station" in line and j + 1 < len(detail_lines):
                    next_line = detail_lines[j + 1]
                    if "miles" in next_line:
                        stations.append(line + " - " + next_line)
                        j += 1

                j += 1

            if len(stations) > 0:
                nearest_stations_all = " | ".join(stations)

    detail_data.append({
        "sector": sector,
        "key_features": key_features,
        "full_description": full_description,
        "nearest_stations_detail": nearest_stations_all
    })

detail_df = pd.DataFrame(detail_data)

df_full = pd.concat([df, detail_df], axis=1)
df_full = df_full[[
    "location",
    "price",
    "size",
    "sector",
    "key_features",
    "gap_station (miles from bank station)",
    "nearest_stations_detail",
    "phone_number",
    "contact_info",
    "description",
    "full_description",
    "links"
]]

df_show = df_full.copy()
show_col = ["full_description", "key_features"]
for col in show_col:
    df_show[col] = df_show[col].apply(
        lambda x: x[:80] + "..." if isinstance(x, str) and len(x) > 80 else x
    )

In [10]:
pd.set_option("display.max_colwidth", None)

display(df_show[[
    "location",
    "price",
    "size",
    "sector",
    "key_features",
    "gap_station (miles from bank station)",
    "nearest_stations_detail",
    "phone_number",
    "contact_info",
    "description",
    "full_description",
    "links"
]])

,location,price,size,sector,key_features,gap_station (miles from bank station),nearest_stations_detail,phone_number,contact_info,description,full_description,links
0,"The Railway Bell, 14 Cawnpore Street, London, SE19 1PF",POA,"2,649 sq. ft.",Pub for sale,"Pub arranged on basement, ground and first floor | First floor is divided into 5...",N/A,Gipsy Hill Station - 0.1 miles | Crystal Palace Station - 0.5 miles | Sydenham Hill Station - 0.7 miles,020 3835 4135,"Marketed by Kalmars Commercial Limited, London","Comprising a late Victorian brick built pub arranged on basement, ground and first floor under a pitched roof, with a garden and storage buildings to the rear. The basement has been used for storing barrels, the ground floor has a pub with toilets to the rear and the first floor is divided into 5...","Comprising a late Victorian brick built pub arranged on basement, ground and fir...",https://www.rightmove.co.uk/properties/174176258
1,"Camden, London, NW1","£50,000",N/A,Restaurant for sale,Spacious | Close to Train Station | High Footfall | Busy Parade | Easy to manage...,N/A,Camden Town Station - 0.1 miles | Mornington Crescent Station - 0.2 miles | Camden Road Station - 0.3 miles,020 3947 5994,"Marketed by GALAXY REAL ESTATE LIMITED, Hayes",GALAXY REAL ESTATE PROUDLY PRESENTS TO THE MARKET.,"""PRIME TAKEAWAY LEASE FOR SALE IN PRIME LOCATION IN CAMDEN"" Discover an exceptio...",https://www.rightmove.co.uk/properties/174258068
2,"Green Lanes, Winchmore Hill, London","£40,000",729 sq. ft.,Takeaway for sale,N/A,N/A,Winchmore Hill Station - 0.3 miles | Grange Park Station - 0.6 miles | Bush Hill Park Station - 1.2 miles,020 3834 8258,"Marketed by Mi Commercial, London","A well situated, ground floor take away premises with private parking. The property is fully equipped and ready to continue as an ongoing concern, or transformation to a new cuisine. The premises is of good specification throughout, including a frameless glass shopfront, air conditioning, a walk ...","LOCATION The premises is ideally situated in a prominent parade on Green Lanes, ...",https://www.rightmove.co.uk/properties/152578379
3,"London, W14","£10,000",N/A,Pub for sale,Spacious | Close to Train Station | High Footfall | Attractive Location | ideal ...,N/A,Kensington Olympia Station - 0.1 miles | Barons Court Station - 0.5 miles | West Kensington Station - 0.5 miles,020 3947 5994,"Marketed by GALAXY REAL ESTATE LIMITED, Hayes",GALAXY REAL ESTATE PROUDLY PRESENTS TO THE MARKET. The currently shut down pub is ready to let but needs cleaning and renovation Currently the premises licence expired; need to apply Ideal location for pub and restaurant,"""PUB TO LET IN PRIME LOCATION IN KENSINGTON."" Welcome to an exceptional opportun...",https://www.rightmove.co.uk/properties/87990147
4,"The Duke of Richmond, 316 Queensbridge Road, Hackney, E8 3NH","£500,000",N/A,N/A,Guide Price Reflects 12% Gross Yield | 20 Year Lease in Place from July 2025 | £...,2.16,N/A,03331 882955,"Marketed by Christie & Co, Pubs & Restaurants","Virtual Freehold. Guide Price Reflects 12% Gross Yield. 20 Year Lease in Place from July 2025. £60,000 per annum rent. Lock up Ground Floor and Basement Property. External Seating Below Heated Canopy",Description The Duke of Richmond is a well-established neighborhood pub and rest...,https://www.rightmove.co.uk/properties/730294825600209
5,"77 Brooksby's Walk, London, E9","£500,000","1,075 sq. ft.",Restaurant for sale,Rare and Unique Opportunity | Development/Conversion Potential (STP) | Suitable ...,3.28,Homerton Station - 0.3 miles | Hackney Central Station - 0.6 miles | Hackney Downs Station - 0.8 miles,01708 201378,"Marketed by Kemsley LLP, Rainham","The property comprises a rare opportunity to acquire a unique investment/development opportunity, comprising a former public convenience most recently used as a bar and restaurant. Internally, the accommodation is currently configured to provide open plan seating with bar and kitchen area. To th...

In [14]:
df_full.duplicated(subset=["location"]).sum()

0

In [16]:
df_full.to_csv("rightmove_full_detail_1.csv", index=False, encoding="utf-8-sig")